In [1]:
###############################################################################
# This version:
#   1) Stores final cosine similarity for each drift_strength (instead of plotting
#      entire time series against batch index).
#   2) Plots drift_strength on the x-axis vs final similarity on the y-axis.
#   3) Demonstrates a Jupyter-friendly structure: one function to collect data,
#      another to plot. You can run `collect_data()` once, then reuse `results`
#      in new cells without re-running everything.
###############################################################################

import os
import random
import torch
import numpy as np
import matplotlib.pyplot as plt

from transformers import AutoTokenizer, AutoModel
from datasets import load_dataset
from tqdm import tqdm
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity

from collections import defaultdict

###############################################################################
# Arguments
###############################################################################
args = {
    "models": ["nlpaueb/sec-bert-base", "bert-base-uncased"],
    "datasets": [
        {
            "name": "yelp_review_full",
            "config": None,
            "split": "train",
            "text_column": "text",
        },
        {
            "name": "wikitext",
            "config": "wikitext-2-raw-v1",
            "split": "train",
            "text_column": "text",
        },
        {"name": "ag_news", "config": None, "split": "train", "text_column": "text"},
    ],
    "max_texts": 1000,
    "batch_size": 64,
    "drift_strengths": [0.0, 0.25, 0.5, 0.75, 1.0],
    "pca_components": 2,
    "output_dir": "results_multidataset",
}

os.makedirs(args["output_dir"], exist_ok=True)


###############################################################################
# Utility Functions
###############################################################################
def batch_generator(data, batch_size=32):
    for i in range(0, len(data), batch_size):
        yield data[i : i + batch_size]


def extract_cls_embeddings(model, tokenizer, texts, device):
    encodings = tokenizer(
        texts, return_tensors="pt", padding=True, truncation=True, max_length=128
    )
    input_ids = encodings["input_ids"].to(device)
    attention_mask = encodings["attention_mask"].to(device)
    with torch.no_grad():
        outputs = model(input_ids, attention_mask=attention_mask)
        cls_embeddings = outputs.last_hidden_state[:, 0, :]
    return cls_embeddings.cpu().numpy()


def introduce_gradual_drift(text_list, fraction_shuffle=0.5):
    new_texts = []
    for txt in text_list:
        words = txt.split()
        if len(words) < 2:
            new_texts.append(txt)
            continue
        k = int(len(words) * fraction_shuffle)
        if k < 1:
            new_texts.append(txt)
            continue
        indices = list(range(len(words)))
        random.shuffle(indices)
        shuffle_indices = indices[:k]
        to_shuffle = [words[i] for i in shuffle_indices]
        random.shuffle(to_shuffle)
        for i, idx in enumerate(shuffle_indices):
            words[idx] = to_shuffle[i]
        new_texts.append(" ".join(words))
    return new_texts


###############################################################################
# DriftDetector Class
###############################################################################
class DriftDetector:
    def __init__(
        self, model, tokenizer, device, batch_generator, args, pca_transform=None
    ):
        self.model = model
        self.tokenizer = tokenizer
        self.device = device
        self.batch_generator = batch_generator
        self.args = args
        self.pca_transform = pca_transform
        self.prototype = None
        self.prototypes = []
        self.cosine_scores = []

    def initialize_baseline(self, texts):
        embeddings = []
        for batch in self.batch_generator(texts, self.args["batch_size"]):
            cls_emb = extract_cls_embeddings(
                self.model, self.tokenizer, batch, self.device
            )
            embeddings.append(cls_emb)
        all_embeddings = np.concatenate(embeddings, axis=0)
        if self.pca_transform is not None:
            all_embeddings = self.pca_transform.transform(all_embeddings)
        self.prototype = np.mean(all_embeddings, axis=0)
        self.prototypes.append(self.prototype)

    def detect_drifts(self, texts):
        for batch in tqdm(self.batch_generator(texts, self.args["batch_size"])):
            batch_embeddings = extract_cls_embeddings(
                self.model, self.tokenizer, batch, self.device
            )
            if self.pca_transform is not None:
                batch_embeddings = self.pca_transform.transform(batch_embeddings)
            mean_emb = batch_embeddings.mean(axis=0, keepdims=True)
            sim = cosine_similarity(mean_emb, [self.prototype])[0][0]
            self.cosine_scores.append(sim)
            self._update_prototype(batch_embeddings)

    def _update_prototype(self, batch_embeddings):
        delta = batch_embeddings - self.prototype
        distances = np.linalg.norm(delta, axis=1)
        weights = np.exp(-distances / 2.0)
        weighted_sum = np.sum(weights[:, None] * delta, axis=0)
        self.prototype += weighted_sum / np.sum(weights)
        self.prototypes.append(self.prototype)


###############################################################################
# 1) Data Collection (store final similarity per drift_strength)
###############################################################################
def collect_data():
    if torch.backends.mps.is_available():
        device = torch.device("mps")
    elif torch.cuda.is_available():
        device = torch.device("cuda")
    else:
        device = torch.device("cpu")
    print("Using device:", device)

    os.makedirs(args["output_dir"], exist_ok=True)
    results = []

    for dataset_info in args["datasets"]:
        dataset_name = dataset_info["name"]
        dataset_config = dataset_info["config"]
        dataset_split = dataset_info["split"]
        text_col = dataset_info["text_column"]

        print(f"\n=== Loading dataset: {dataset_name} ===")
        ds = load_dataset(dataset_name, dataset_config, split=dataset_split)
        texts = ds[text_col]
        texts = list(texts)
        random.shuffle(texts)
        if args["max_texts"] > 0 and len(texts) > args["max_texts"]:
            texts = texts[: args["max_texts"]]

        half_point = len(texts) // 2
        baseline_texts = texts[:half_point]

        for model_name in args["models"]:
            print(f"\n--- Using Model: {model_name} ---")
            tokenizer = AutoTokenizer.from_pretrained(model_name)
            model = AutoModel.from_pretrained(model_name)
            model.to(device)
            model.eval()

            baseline_embs = []
            for b in batch_generator(baseline_texts, args["batch_size"]):
                emb_b = extract_cls_embeddings(model, tokenizer, b, device)
                baseline_embs.append(emb_b)
            baseline_embs = np.concatenate(baseline_embs, axis=0)

            pca = PCA(n_components=args["pca_components"])
            pca.fit(baseline_embs)

            for drift_strength in args["drift_strengths"]:
                print(f"\nSimulating drift with strength={drift_strength}")
                drifted_texts = introduce_gradual_drift(
                    texts[half_point:], fraction_shuffle=drift_strength
                )
                test_texts = baseline_texts + drifted_texts

                detector_no_pca = DriftDetector(
                    model=model,
                    tokenizer=tokenizer,
                    device=device,
                    batch_generator=batch_generator,
                    args=args,
                    pca_transform=None,
                )
                detector_no_pca.initialize_baseline(baseline_texts)
                detector_no_pca.detect_drifts(test_texts)
                final_sim_no_pca = detector_no_pca.cosine_scores[-1]

                detector_pca = DriftDetector(
                    model=model,
                    tokenizer=tokenizer,
                    device=device,
                    batch_generator=batch_generator,
                    args=args,
                    pca_transform=pca,
                )
                detector_pca.initialize_baseline(baseline_texts)
                detector_pca.detect_drifts(test_texts)
                final_sim_pca = detector_pca.cosine_scores[-1]

                results.append(
                    {
                        "dataset": dataset_name,
                        "model": model_name,
                        "drift_strength": drift_strength,
                        "pca": False,
                        "final_similarity": final_sim_no_pca,
                    }
                )

                results.append(
                    {
                        "dataset": dataset_name,
                        "model": model_name,
                        "drift_strength": drift_strength,
                        "pca": True,
                        "final_similarity": final_sim_pca,
                    }
                )

    print("\nData collection done!")
    return results


# Collect
results = collect_data()

/Users/jasper.bruin/miniconda3/envs/driftwatch/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: mps

=== Loading dataset: yelp_review_full ===

--- Using Model: nlpaueb/sec-bert-base ---

Simulating drift with strength=0.0


16it [00:06,  2.63it/s]
16it [00:06,  2.59it/s]



Simulating drift with strength=0.25


16it [00:06,  2.58it/s]
16it [00:06,  2.61it/s]


...


Simulating drift with strength=0.75


16it [00:05,  2.90it/s]
16it [00:05,  2.91it/s]



Simulating drift with strength=1.0


16it [00:05,  2.91it/s]
16it [00:05,  2.91it/s]


Data collection done!


In [2]:
###############################################################################
# 2) Plotting: drift_strength on x-axis, final similarity on y-axis
###############################################################################
def plot_data(results):
    grouped = defaultdict(list)
    for r in results:
        key = (r["dataset"], r["model"], r["pca"])
        grouped[key].append(r)

    # We'll make one figure per (dataset, model) pair,
    # with two lines on the same axes: PCA vs No PCA
    # x-axis: drift_strength, y-axis: final_similarity
    for (dataset_name, model_name, pca_flag), group_vals in grouped.items():
        # We'll handle plotting in larger groups, so only do it once per (dataset, model)
        # Skip if we've already plotted this combination
        # We can gather no_pca and pca in one figure by ignoring pca_flag in final step

        # Check if there's a matching "pca=False" & "pca=True" group
        pass

    # Let's group again by (dataset, model) so we can plot pca/no-pca lines together
    grouped_dm = defaultdict(lambda: {"pca": [], "no_pca": []})
    for r in results:
        dm_key = (r["dataset"], r["model"])
        if r["pca"]:
            grouped_dm[dm_key]["pca"].append(r)
        else:
            grouped_dm[dm_key]["no_pca"].append(r)

    for (dataset_name, model_name), subdict in grouped_dm.items():
        # Sort each list by drift_strength so lines look nice
        no_pca_list = sorted(subdict["no_pca"], key=lambda x: x["drift_strength"])
        pca_list = sorted(subdict["pca"], key=lambda x: x["drift_strength"])

        x_no_pca = [g["drift_strength"] for g in no_pca_list]
        y_no_pca = [g["final_similarity"] for g in no_pca_list]
        x_pca = [g["drift_strength"] for g in pca_list]
        y_pca = [g["final_similarity"] for g in pca_list]

        plt.figure(figsize=(7, 5))
        plt.plot(x_no_pca, y_no_pca, marker="o", label="No PCA")
        plt.plot(x_pca, y_pca, marker="s", label="PCA")
        plt.title(f"{dataset_name} | {model_name}")
        plt.xlabel("Drift Strength")
        plt.ylabel("Final Cosine Similarity")
        plt.grid(True, linestyle="--", alpha=0.5)
        plt.legend()

        model_name_safe = model_name.replace("/", "_")
        fname = f"{dataset_name}_{model_name_safe}_final_vs_drift.png"
        save_path = os.path.join(args["output_dir"], fname)
        plt.savefig(save_path)
        plt.close()
        print(f"Saved final-sim-vs-drift plot: {save_path}")


# Plot
plot_data(results)
print("All done.")

Saved final-sim-vs-drift plot: results_multidataset/yelp_review_full_nlpaueb_sec-bert-base_final_vs_drift.png
Saved final-sim-vs-drift plot: results_multidataset/yelp_review_full_bert-base-uncased_final_vs_drift.png
Saved final-sim-vs-drift plot: results_multidataset/wikitext_nlpaueb_sec-bert-base_final_vs_drift.png
Saved final-sim-vs-drift plot: results_multidataset/wikitext_bert-base-uncased_final_vs_drift.png
Saved final-sim-vs-drift plot: results_multidataset/ag_news_nlpaueb_sec-bert-base_final_vs_drift.png
Saved final-sim-vs-drift plot: results_multidataset/ag_news_bert-base-uncased_final_vs_drift.png
All done.


In [5]:
###############################################################################
# 2) Plotting: drift_strength on x-axis vs. final similarity on y-axis
###############################################################################
def plot_data(results):
    """
    We group by (dataset, model) so we can plot two lines:
    - 'No PCA'
    - 'PCA'
    """
    from collections import defaultdict

    grouped_dm = defaultdict(lambda: {"pca": [], "no_pca": []})
    for r in results:
        dm_key = (r["dataset"], r["model"])
        if r["pca"]:
            grouped_dm[dm_key]["pca"].append(r)
        else:
            grouped_dm[dm_key]["no_pca"].append(r)

    # For each dataset/model pair, plot two lines: PCA vs. no PCA
    for (dataset_name, model_name), subdict in grouped_dm.items():
        no_pca_list = sorted(subdict["no_pca"], key=lambda x: x["drift_strength"])
        pca_list = sorted(subdict["pca"], key=lambda x: x["drift_strength"])

        x_no_pca = [g["drift_strength"] for g in no_pca_list]
        y_no_pca = [g["final_similarity"] for g in no_pca_list]
        x_pca = [g["drift_strength"] for g in pca_list]
        y_pca = [g["final_similarity"] for g in pca_list]

        plt.figure(figsize=(7, 5))
        plt.plot(x_no_pca, y_no_pca, marker="o", label="No PCA")
        plt.plot(x_pca, y_pca, marker="s", label="PCA")

        plt.title(f"Final Similarity vs. Drift Strength\n{dataset_name} | {model_name}")
        plt.xlabel("Drift Strength")
        plt.ylabel("Final Cosine Similarity")
        plt.grid(True, linestyle="--", alpha=0.5)
        plt.legend()

        model_name_safe = model_name.replace("/", "_")
        fname = f"{dataset_name}_{model_name_safe}_final_vs_drift2.png"
        save_path = os.path.join(args["output_dir"], fname)
        plt.savefig(save_path)
        plt.close()
        print(f"Saved: {save_path}")


# Finally, plot the results
plot_data(results)
print("All done.")

Saved: results_multidataset/yelp_review_full_nlpaueb_sec-bert-base_final_vs_drift2.png
Saved: results_multidataset/yelp_review_full_bert-base-uncased_final_vs_drift2.png
Saved: results_multidataset/wikitext_nlpaueb_sec-bert-base_final_vs_drift2.png
Saved: results_multidataset/wikitext_bert-base-uncased_final_vs_drift2.png
Saved: results_multidataset/ag_news_nlpaueb_sec-bert-base_final_vs_drift2.png
Saved: results_multidataset/ag_news_bert-base-uncased_final_vs_drift2.png
All done.
